In [9]:
import pandas as pd
import re
import os

# 1. 정규식 패턴 정의
pattern_count = r"몇|개입|개수|개인"
# 🚨 [추가] 선택지에서 "숫자+단위"를 잡아내는 강력한 정규식 (이상/미만, 권, 캔 등 포함)
pattern_count_choice = r"\d+\s*(개|권|장|병|캔|봉지|통|박스|이상|미만)"
pattern_materials = r"플라스틱|페트|종이|박스|골판지|유리|금속|철|캔|알루미늄|비닐|봉지|팩|스티로폼"
pattern_ask_mat = r"재질은|분류는|무슨\s*재질|어떤\s*재질|재질이\s*무엇|무엇으로\s*만들어|종류는"
pattern_ask_obj = r"무엇|어느|어떤\s*것"

# 🚨 [수정] 질문(text) 하나만 받던 것을, 행(Row) 전체를 받도록 변경
def classify_question_advanced(row):
    q_text = str(row['question']) if pd.notna(row['question']) else "Other"
    
    # a, b, c, d 선택지를 하나의 텍스트로 결합
    choices_text = f"{row.get('a','')} {row.get('b','')} {row.get('c','')} {row.get('d','')}"
    
    # 질문과 선택지 각각에서 카운팅 패턴 확인
    is_count_q = bool(re.search(pattern_count, q_text))
    is_count_c = bool(re.search(pattern_count_choice, choices_text))
    is_ocr = bool(re.search(r"적힌|글자|라벨|로고|년|표시|문구|숫자", q_text))
    
    # YOLO 배정 로직에 'is_ocr이 아닐 때만' 이라는 조건 추가
    if (is_count_q or is_count_c) and not is_ocr:
        # train_0096.jpg 처럼 질문이 "무엇"으로 끝나면 VLM으로 우회
        if "무엇" in q_text:
            return "VLM"
        return "YOLO"
    return "VLM"

def process_and_save_csv(file_path):
    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return
        
    df = pd.read_csv(file_path)
    
    # 🚨 [수정] apply 함수에 axis=1 옵션을 주어 row(행) 단위로 함수에 던져줌
    df['class'] = df.apply(classify_question_advanced, axis=1)
    
    new_file_path = file_path.replace('.csv', '_class.csv')
    df.to_csv(new_file_path, index=False, encoding='utf-8-sig')
    print(f"저장 완료: {new_file_path} (총 {len(df)}행)")

if __name__ == "__main__":
    target_files = ['train.csv', 'dev.csv', 'test.csv']
    
    for file in target_files:
        process_and_save_csv(file)
        
    print("모든 파일의 섬세한 분류 및 저장이 완료되었습니다.")

저장 완료: train_class.csv (총 5073행)
저장 완료: dev_class.csv (총 4413행)
저장 완료: test_class.csv (총 5074행)
모든 파일의 섬세한 분류 및 저장이 완료되었습니다.
